
# ot4.ipynb

`ot4`는 기존 `ot3`의 동시 판정 구조를 유지하면서, 싸움 판정을 더 정교하게 만들기 위한 버전입니다.

핵심 변경점:

1. 모든 결과를 `C:\Users\우성한\outputs\prediction\aa` 폴더 안에 저장합니다.
2. 기존 CF2 객체 탐지 JSON을 생성합니다.
3. 새 중간 JSON인 `aa_features`를 생성합니다.
   - 사람 중심점 변화
   - 간단한 track id
   - 사람 간 거리 변화
   - 근접 상태에서의 움직임
   - 방향 변화
   - 위치 역전
   - bbox overlap
   - 5초 슬라이딩 윈도우 fight/stalking score
4. 최종적으로 `bd4_improved` 행동 판정과 `cf6_like` 이륜차 판정을 동시에 저장합니다.

입력 영상 루트:

```python
C:\Users\우성한\Sample (1)\Sample\01.원천데이터
```


In [1]:

# Cell 1. 경로 및 전체 설정
from pathlib import Path
import os, json, math, time, gc
from collections import Counter, defaultdict

VIDEO_ROOT = Path(r"C:\Users\우성한\Sample (1)\Sample\01.원천데이터")
OUTPUT_ROOT = Path(r"C:\Users\우성한\outputs\prediction")

# 모든 결과는 aa 폴더 아래 저장
AA_DIR = OUTPUT_ROOT / "aa"
AA_CF2_DIR = AA_DIR / "cf2"
AA_FEATURE_DIR = AA_DIR / "aa_features"
AA_BD4_DIR = AA_DIR / "bd4_improved"
AA_CF6_DIR = AA_DIR / "cf6_like"
AA_FINAL_DIR = AA_DIR / "final_ot4"
AA_DEBUG_DIR = AA_DIR / "debug"

for d in [AA_DIR, AA_CF2_DIR, AA_FEATURE_DIR, AA_BD4_DIR, AA_CF6_DIR, AA_FINAL_DIR, AA_DEBUG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

VIDEO_EXTS = {
    ".mp4", ".avi", ".mov", ".mkv", ".wmv", ".mpeg", ".mpg", ".m4v",
    ".MP4", ".AVI", ".MOV", ".MKV", ".WMV", ".MPEG", ".MPG", ".M4V"
}

# 테스트할 때만 숫자로 바꾸고, 전체 처리 시 None
MAX_VIDEOS = None
MAX_JSON_FILES = None
MAX_FRAMES_PER_VIDEO = None
# MAX_VIDEOS = 3
# MAX_JSON_FILES = 3
# MAX_FRAMES_PER_VIDEO = 160

YOLO_MODEL_PATH = "yolov8n.pt"
CF2_FRAME_INTERVAL = 10   # 싸움 움직임을 보려면 15보다 10이 조금 유리함
PERSON_MIN_CONF = 0.15
VEHICLE_MIN_CONF = 0.10
TRACK_MAX_DIST = 130
WINDOW_SECONDS = 5.0
WINDOW_STRIDE_SECONDS = 1.5

# 판정 threshold. 결과가 너무 보수적이면 FIGHT_THRESHOLD를 0.34 정도로 낮추고,
# 너무 포괄적이면 0.42 이상으로 올리면 됨.
FIGHT_THRESHOLD = 0.38
STALKING_THRESHOLD = 0.36

TWO_WHEELER_CLASSES = {"bicycle", "motorcycle"}
PERSON_CLASSES = {"person"}

print("VIDEO_ROOT exists:", VIDEO_ROOT.exists(), VIDEO_ROOT)
print("AA_DIR:", AA_DIR)

if not VIDEO_ROOT.exists():
    raise FileNotFoundError(f"원본 영상 루트를 찾지 못했습니다: {VIDEO_ROOT}")


VIDEO_ROOT exists: True C:\Users\우성한\Sample (1)\Sample\01.원천데이터
AA_DIR: C:\Users\우성한\outputs\prediction\aa


In [2]:

# Cell 2. 의존성 확인
missing = []

try:
    import cv2
    print("cv2 OK")
except Exception as e:
    missing.append("opencv-python")
    print("cv2 import failed:", repr(e))

try:
    import numpy as np
    print("numpy OK")
except Exception as e:
    missing.append("numpy")
    print("numpy import failed:", repr(e))

try:
    from tqdm import tqdm
    print("tqdm OK")
except Exception as e:
    missing.append("tqdm")
    print("tqdm import failed:", repr(e))

try:
    from ultralytics import YOLO
    print("ultralytics OK")
except Exception as e:
    YOLO = None
    missing.append("ultralytics")
    print("ultralytics import failed:", repr(e))

if missing:
    print("\n설치 필요 후보:", sorted(set(missing)))
    print("예: pip install opencv-python numpy tqdm ultralytics")


cv2 OK
numpy OK
tqdm OK
ultralytics OK


In [3]:

# Cell 3. 원천데이터 전체 영상 탐색
video_paths = [p for p in VIDEO_ROOT.rglob("*") if p.is_file() and p.suffix in VIDEO_EXTS]
video_paths = sorted(video_paths)

if MAX_VIDEOS is not None:
    video_paths = video_paths[:MAX_VIDEOS]

print("찾은 영상 수:", len(video_paths))
for p in video_paths[:30]:
    print(p)

if len(video_paths) == 0:
    raise RuntimeError("원천데이터 경로 안에서 영상 파일을 찾지 못했습니다.")


찾은 영상 수: 22
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 구역 내 지속 배회\lo_e1281\lo_e1281_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 구역 내 지속 배회\lo_e1281\lo_e1281_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 인물을 뒤따라가며 배회\fp_e1520\fp_e1520_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 인물을 뒤따라가며 배회\fp_e1520\fp_e1520_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\싸움\신체적 충돌을 동반한 싸움\ph_e2063\ph_e2063_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\싸움\신체적 충돌을 동반한 싸움\ph_e2063\ph_e2063_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\오토바이 인도 주행\ms_e3500\ms_e3500_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\오토바이 인도 주행\ms_e3500\ms_e3500_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\자전거 인도 주행\bs_e4387\bs_e4387_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\자전거 인도 주행\bs_e4387\bs_e4387_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\전동킥보드 인도 주행\es_e3768\es_e3768_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데

In [4]:

# Cell 4. CF2 생성/확인
if YOLO is None:
    raise RuntimeError("ultralytics YOLO를 사용할 수 없습니다.")

cf2_yolo_model = YOLO(YOLO_MODEL_PATH)
print("YOLO model loaded:", YOLO_MODEL_PATH)


def safe_stem_from_path(path: Path, root: Path):
    rel = path.relative_to(root)
    return "__".join(rel.with_suffix("").parts)


def video_meta(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return {"fps": None, "width": None, "height": None, "frame_count": None}
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return {
        "fps": float(fps) if fps and fps > 0 else None,
        "width": width,
        "height": height,
        "frame_count": frame_count,
    }


def create_cf2_from_video(video_path, output_json_path, frame_interval=10):
    meta = video_meta(video_path)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"영상 열기 실패: {video_path}")

    frames = []
    frame_idx = 0
    sampled_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if MAX_FRAMES_PER_VIDEO is not None and frame_idx >= MAX_FRAMES_PER_VIDEO:
            break

        if frame_idx % frame_interval == 0:
            result = cf2_yolo_model(frame, verbose=False)[0]
            objects = []
            classes = {}

            if result.boxes is not None:
                for box in result.boxes:
                    cls_id = int(box.cls[0])
                    cls_name = cf2_yolo_model.names[cls_id]
                    conf = float(box.conf[0])
                    bbox = [round(float(v), 2) for v in box.xyxy[0].tolist()]
                    objects.append({"class": cls_name, "confidence": round(conf, 4), "bbox": bbox})
                    classes[cls_name] = classes.get(cls_name, 0) + 1

            timestamp_sec = None
            if meta.get("fps"):
                timestamp_sec = round(frame_idx / meta["fps"], 3)

            frames.append({
                "frame_id": frame_idx,
                "sampled_id": sampled_idx,
                "timestamp_sec": timestamp_sec,
                "object_count": len(objects),
                "classes": classes,
                "objects": objects,
            })
            sampled_idx += 1

        frame_idx += 1

    cap.release()

    summary_classes = {}
    for fr in frames:
        for cls_name, count in fr["classes"].items():
            summary_classes[cls_name] = summary_classes.get(cls_name, 0) + count

    data = {
        "video_name": video_path.name,
        "video_path": str(video_path),
        "relative_video_path": str(video_path.relative_to(VIDEO_ROOT)),
        "meta": {
            **meta,
            "cf2_frame_interval": frame_interval,
            "sampled_frames": len(frames),
            "actually_read_frames": frame_idx,
        },
        "summary": {
            "total_frames_read": frame_idx,
            "sampled_frames": len(frames),
            "classes": summary_classes,
        },
        "frames": frames,
    }

    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    return data


created, used_existing, failed = 0, 0, []

for video_path in tqdm(video_paths):
    stem = safe_stem_from_path(video_path, VIDEO_ROOT)
    output_json_path = AA_CF2_DIR / f"{stem}.json"

    if output_json_path.exists():
        used_existing += 1
        continue

    try:
        create_cf2_from_video(video_path, output_json_path, frame_interval=CF2_FRAME_INTERVAL)
        created += 1
    except Exception as e:
        failed.append({"video": str(video_path), "error": repr(e)})

print("\n================================")
print("AA CF2 생성/확인 완료")
print("전체 영상:", len(video_paths))
print("생성:", created)
print("기존 사용:", used_existing)
print("실패:", len(failed))
print("저장 위치:", AA_CF2_DIR)
print("================================")

if failed:
    print("\n실패 목록 일부:")
    for item in failed[:20]:
        print(item)


YOLO model loaded: yolov8n.pt


100%|██████████████████████████████████████████████████████████████████████████████████| 22/22 [02:05<00:00,  5.73s/it]


AA CF2 생성/확인 완료
전체 영상: 22
생성: 22
기존 사용: 0
실패: 0
저장 위치: C:\Users\우성한\outputs\prediction\aa\cf2


In [5]:

# Cell 5. aa_features 생성용 함수

def clamp(v, lo=0.0, hi=1.0):
    return max(lo, min(hi, v))


def bbox_center(bbox):
    x1, y1, x2, y2 = bbox
    return ((x1 + x2) / 2, (y1 + y2) / 2)


def bbox_area(bbox):
    x1, y1, x2, y2 = bbox
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


def bbox_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    union = bbox_area(a) + bbox_area(b) - inter
    return inter / union if union > 0 else 0.0


def dist(p1, p2):
    return math.sqrt((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2)


def vec(p1, p2):
    return (p2[0] - p1[0], p2[1] - p1[1])


def vec_norm(v):
    return math.sqrt(v[0] ** 2 + v[1] ** 2)


def cosine(v1, v2):
    n1, n2 = vec_norm(v1), vec_norm(v2)
    if n1 < 1e-6 or n2 < 1e-6:
        return None
    return (v1[0] * v2[0] + v1[1] * v2[1]) / (n1 * n2)


def extract_person_detections(frame):
    persons = []
    for obj in frame.get("objects", []):
        if obj.get("class") in PERSON_CLASSES and obj.get("confidence", 0) >= PERSON_MIN_CONF:
            bbox = obj.get("bbox")
            if bbox and len(bbox) == 4:
                persons.append({
                    "bbox": bbox,
                    "center": bbox_center(bbox),
                    "area": bbox_area(bbox),
                    "confidence": obj.get("confidence", 0),
                })
    return persons


def extract_vehicle_detections(frame):
    vehicles = []
    for obj in frame.get("objects", []):
        if obj.get("class") in TWO_WHEELER_CLASSES and obj.get("confidence", 0) >= VEHICLE_MIN_CONF:
            bbox = obj.get("bbox")
            if bbox and len(bbox) == 4:
                vehicles.append({
                    "class": obj.get("class"),
                    "bbox": bbox,
                    "center": bbox_center(bbox),
                    "area": bbox_area(bbox),
                    "confidence": obj.get("confidence", 0),
                })
    return vehicles


def assign_tracks(frames):
    """간단한 greedy tracker. 외부 tracking 라이브러리 없이 person bbox 흐름만 대략 연결."""
    next_track_id = 1
    active = {}  # track_id -> last center
    active_last_seen = {}
    tracked_frames = []

    for idx, frame in enumerate(frames):
        persons = extract_person_detections(frame)
        used_tracks = set()

        # 각 person을 가장 가까운 기존 track에 배정
        for p in persons:
            best_tid, best_d = None, None
            for tid, center in active.items():
                if tid in used_tracks:
                    continue
                if idx - active_last_seen.get(tid, -999) > 2:
                    continue
                d = dist(p["center"], center)
                if d <= TRACK_MAX_DIST and (best_d is None or d < best_d):
                    best_tid, best_d = tid, d

            if best_tid is None:
                best_tid = next_track_id
                next_track_id += 1

            p["track_id"] = best_tid
            p["track_match_dist"] = round(best_d, 3) if best_d is not None else None
            active[best_tid] = p["center"]
            active_last_seen[best_tid] = idx
            used_tracks.add(best_tid)

        vehicles = extract_vehicle_detections(frame)
        tracked_frames.append({
            "frame_id": frame.get("frame_id"),
            "sampled_id": frame.get("sampled_id"),
            "timestamp_sec": frame.get("timestamp_sec"),
            "persons": persons,
            "vehicles": vehicles,
            "person_count": len(persons),
            "vehicle_count": len(vehicles),
        })

    return tracked_frames


def add_motion_features(tracked_frames):
    last_by_track = {}
    prev_velocity = {}

    for fr in tracked_frames:
        for p in fr["persons"]:
            tid = p["track_id"]
            prev = last_by_track.get(tid)

            if prev is None:
                p["motion"] = 0.0
                p["velocity"] = [0.0, 0.0]
                p["direction_change"] = 0.0
            else:
                v = vec(prev["center"], p["center"])
                m = vec_norm(v)
                old_v = prev_velocity.get(tid)
                c = cosine(old_v, v) if old_v is not None else None
                # direction_change: 0이면 방향 유지, 1이면 반대/급변에 가까움
                dc = 0.0 if c is None else clamp((1 - c) / 2)
                p["motion"] = round(m, 3)
                p["velocity"] = [round(v[0], 3), round(v[1], 3)]
                p["direction_change"] = round(dc, 3)
                prev_velocity[tid] = v

            last_by_track[tid] = {"center": p["center"], "frame_id": fr["frame_id"]}

    return tracked_frames


def pair_key(a, b):
    return tuple(sorted([a, b]))


def compute_pair_interactions(tracked_frames, width=None, height=None):
    """프레임별 사람쌍 상호작용 특징 생성."""
    diag = math.sqrt((width or 1920) ** 2 + (height or 1080) ** 2)
    close_dist = diag * 0.075  # 대략 1920x1080 기준 165px 근처
    near_dist = diag * 0.12

    prev_pair_info = {}

    for fr in tracked_frames:
        persons = fr["persons"]
        interactions = []

        for i in range(len(persons)):
            for j in range(i + 1, len(persons)):
                a, b = persons[i], persons[j]
                d = dist(a["center"], b["center"])
                iou = bbox_iou(a["bbox"], b["bbox"])
                avg_motion = (a.get("motion", 0) + b.get("motion", 0)) / 2
                max_motion = max(a.get("motion", 0), b.get("motion", 0))
                avg_dir_change = (a.get("direction_change", 0) + b.get("direction_change", 0)) / 2

                key = pair_key(a["track_id"], b["track_id"])
                prev = prev_pair_info.get(key)
                distance_delta = 0.0
                approach_or_separate = 0
                position_swap = 0
                same_direction = 0

                if prev is not None:
                    distance_delta = d - prev["distance"]
                    if abs(distance_delta) > close_dist * 0.20:
                        approach_or_separate = 1

                    # x축 상대 위치가 바뀌면 서로 뒤섞였을 가능성
                    prev_order = prev["x_order"]
                    now_order = a["center"][0] < b["center"][0]
                    if prev_order != now_order and d < near_dist:
                        position_swap = 1

                    ca = cosine(a.get("velocity", [0, 0]), b.get("velocity", [0, 0]))
                    if ca is not None and ca > 0.65:
                        same_direction = 1

                close_flag = int(d <= close_dist)
                near_flag = int(d <= near_dist)
                close_motion_flag = int(close_flag and avg_motion > close_dist * 0.12)
                overlap_flag = int(iou > 0.015)

                item = {
                    "track_pair": list(key),
                    "distance": round(d, 3),
                    "distance_norm": round(d / diag, 5),
                    "distance_delta": round(distance_delta, 3),
                    "close": close_flag,
                    "near": near_flag,
                    "iou": round(iou, 4),
                    "overlap": overlap_flag,
                    "avg_motion": round(avg_motion, 3),
                    "max_motion": round(max_motion, 3),
                    "avg_direction_change": round(avg_dir_change, 3),
                    "close_motion": close_motion_flag,
                    "approach_or_separate": approach_or_separate,
                    "position_swap": position_swap,
                    "same_direction": same_direction,
                }
                interactions.append(item)

                prev_pair_info[key] = {
                    "distance": d,
                    "x_order": a["center"][0] < b["center"][0],
                }

        fr["interactions"] = interactions
        fr["interaction_count"] = len(interactions)
        fr["close_pair_count"] = sum(x["close"] for x in interactions)
        fr["close_motion_pair_count"] = sum(x["close_motion"] for x in interactions)
        fr["overlap_pair_count"] = sum(x["overlap"] for x in interactions)
        fr["position_swap_count"] = sum(x["position_swap"] for x in interactions)
        fr["same_direction_pair_count"] = sum(x["same_direction"] for x in interactions)

    return tracked_frames


def window_indices(n, window_n, stride_n):
    if n <= 0:
        return []
    window_n = max(2, min(window_n, n))
    stride_n = max(1, stride_n)
    out = []
    start = 0
    while start < n:
        end = min(n, start + window_n)
        out.append((start, end))
        if end == n:
            break
        start += stride_n
    return out


def window_score(items):
    n = len(items)
    if n == 0:
        return {}

    multi_rate = sum(1 for f in items if f["person_count"] >= 2) / n
    person_rate = sum(1 for f in items if f["person_count"] >= 1) / n
    close_rate = sum(1 for f in items if f["close_pair_count"] > 0) / n
    close_motion_rate = sum(1 for f in items if f["close_motion_pair_count"] > 0) / n
    overlap_rate = sum(1 for f in items if f["overlap_pair_count"] > 0) / n
    swap_rate = sum(1 for f in items if f["position_swap_count"] > 0) / n
    same_dir_rate = sum(1 for f in items if f["same_direction_pair_count"] > 0) / n
    vehicle_rate = sum(1 for f in items if f["vehicle_count"] > 0) / n

    # 프레임 내 사람들의 평균 움직임/방향변화
    motions = []
    dir_changes = []
    for f in items:
        for p in f["persons"]:
            motions.append(p.get("motion", 0))
            dir_changes.append(p.get("direction_change", 0))
    avg_motion = sum(motions) / len(motions) if motions else 0.0
    avg_dir_change = sum(dir_changes) / len(dir_changes) if dir_changes else 0.0

    # 접근/분리 반복성
    approach_sep_rate = sum(
        1 for f in items if any(x.get("approach_or_separate") for x in f.get("interactions", []))
    ) / n

    # 싸움: 근접 + 움직임 + 방향 급변 + 위치 뒤섞임 중심
    fight_raw = (
        0.12 * multi_rate +
        0.14 * close_rate +
        0.24 * close_motion_rate +
        0.16 * approach_sep_rate +
        0.14 * clamp(avg_dir_change / 0.45) +
        0.10 * overlap_rate +
        0.10 * swap_rate
    )

    # 같은 방향으로 안정적으로 이동하면 fight보다는 stalking/동행에 가까움
    stable_follow_penalty = 0.14 * same_dir_rate if close_motion_rate < 0.35 else 0.05 * same_dir_rate
    fight_score = clamp(fight_raw - stable_follow_penalty)

    # 스토킹: 사람 지속 + 같은 방향 이동 + 거리 안정 + fight 요소는 낮음
    stalking_raw = (
        0.25 * person_rate +
        0.20 * multi_rate +
        0.20 * same_dir_rate +
        0.15 * approach_sep_rate +
        0.10 * clamp(avg_motion / 80.0) +
        0.10 * (1.0 - overlap_rate)
    )
    stalking_penalty = 0.20 * close_motion_rate + 0.10 * swap_rate
    stalking_score = clamp(stalking_raw - stalking_penalty)

    return {
        "frames": n,
        "person_rate": round(person_rate, 4),
        "multi_rate": round(multi_rate, 4),
        "close_rate": round(close_rate, 4),
        "close_motion_rate": round(close_motion_rate, 4),
        "overlap_rate": round(overlap_rate, 4),
        "swap_rate": round(swap_rate, 4),
        "same_direction_rate": round(same_dir_rate, 4),
        "approach_separate_rate": round(approach_sep_rate, 4),
        "vehicle_rate": round(vehicle_rate, 4),
        "avg_motion": round(avg_motion, 4),
        "avg_direction_change": round(avg_dir_change, 4),
        "fight_score": round(fight_score, 4),
        "stalking_score": round(stalking_score, 4),
    }


def build_aa_features(cf2_data):
    frames = cf2_data.get("frames", [])
    meta = cf2_data.get("meta", {})
    fps = meta.get("fps") or 30.0
    frame_interval = meta.get("cf2_frame_interval") or CF2_FRAME_INTERVAL
    width = meta.get("width") or 1920
    height = meta.get("height") or 1080

    tracked_frames = assign_tracks(frames)
    tracked_frames = add_motion_features(tracked_frames)
    tracked_frames = compute_pair_interactions(tracked_frames, width=width, height=height)

    samples_per_second = fps / frame_interval if frame_interval else 2.0
    window_n = max(3, int(round(WINDOW_SECONDS * samples_per_second)))
    stride_n = max(1, int(round(WINDOW_STRIDE_SECONDS * samples_per_second)))

    windows = []
    for start, end in window_indices(len(tracked_frames), window_n, stride_n):
        ws = window_score(tracked_frames[start:end])
        if not ws:
            continue
        start_ts = tracked_frames[start].get("timestamp_sec")
        end_ts = tracked_frames[end - 1].get("timestamp_sec")
        windows.append({
            "start_index": start,
            "end_index": end - 1,
            "start_frame_id": tracked_frames[start].get("frame_id"),
            "end_frame_id": tracked_frames[end - 1].get("frame_id"),
            "start_sec": start_ts,
            "end_sec": end_ts,
            **ws,
        })

    peak_fight = max(windows, key=lambda x: x.get("fight_score", 0), default=None)
    peak_stalking = max(windows, key=lambda x: x.get("stalking_score", 0), default=None)

    overall = window_score(tracked_frames)

    data = {
        "video_name": cf2_data.get("video_name"),
        "video_path": cf2_data.get("video_path"),
        "relative_video_path": cf2_data.get("relative_video_path"),
        "meta": {
            **meta,
            "window_seconds": WINDOW_SECONDS,
            "window_stride_seconds": WINDOW_STRIDE_SECONDS,
            "window_n_sampled_frames": window_n,
            "stride_n_sampled_frames": stride_n,
            "fight_threshold": FIGHT_THRESHOLD,
            "stalking_threshold": STALKING_THRESHOLD,
        },
        "overall_features": overall,
        "peak_fight_window": peak_fight,
        "peak_stalking_window": peak_stalking,
        "windows": windows,
        "frames": tracked_frames,
    }
    return data


In [6]:

# Cell 6. aa_features 전체 생성
cf2_json_paths = sorted(AA_CF2_DIR.rglob("*.json"))
if MAX_JSON_FILES is not None:
    cf2_json_paths = cf2_json_paths[:MAX_JSON_FILES]

print("CF2 json 수:", len(cf2_json_paths))
if len(cf2_json_paths) == 0:
    raise RuntimeError("AA_CF2_DIR 안에 CF2 json이 없습니다. Cell 4를 먼저 확인하세요.")

created, used_existing, failed = 0, 0, []

for json_path in tqdm(cf2_json_paths):
    out_path = AA_FEATURE_DIR / json_path.name
    if out_path.exists():
        used_existing += 1
        continue
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            cf2_data = json.load(f)
        aa_data = build_aa_features(cf2_data)
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(aa_data, f, ensure_ascii=False, indent=2)
        created += 1
    except Exception as e:
        failed.append({"json": str(json_path), "error": repr(e)})

print("\n================================")
print("aa_features 생성/확인 완료")
print("전체 json:", len(cf2_json_paths))
print("생성:", created)
print("기존 사용:", used_existing)
print("실패:", len(failed))
print("저장 위치:", AA_FEATURE_DIR)
print("================================")

if failed:
    print("\n실패 목록 일부:")
    for item in failed[:20]:
        print(item)


CF2 json 수: 22


100%|██████████████████████████████████████████████████████████████████████████████████| 22/22 [00:00<00:00, 82.44it/s]


aa_features 생성/확인 완료
전체 json: 22
생성: 22
기존 사용: 0
실패: 0
저장 위치: C:\Users\우성한\outputs\prediction\aa\aa_features


In [7]:

# Cell 7. improved bd4 행동 판정 함수

def classify_action_from_aa(aa_data):
    peak_fight = aa_data.get("peak_fight_window") or {}
    peak_stalking = aa_data.get("peak_stalking_window") or {}
    overall = aa_data.get("overall_features") or {}

    fight_score = peak_fight.get("fight_score", 0.0)
    stalking_score = peak_stalking.get("stalking_score", 0.0)

    # fight는 peak window 중심으로 판단한다.
    # 단, 사람이 2명 이상 거의 없으면 fight 제외.
    fight_multi_ok = peak_fight.get("multi_rate", 0) >= 0.25
    fight_interaction_ok = (
        peak_fight.get("close_motion_rate", 0) >= 0.18 or
        peak_fight.get("approach_separate_rate", 0) >= 0.25 or
        peak_fight.get("swap_rate", 0) >= 0.12 or
        peak_fight.get("overlap_rate", 0) >= 0.12
    )

    # stalking은 fight보다 움직임이 덜 뒤섞이고, 같은 방향성이 어느 정도 있어야 함.
    stalking_person_ok = peak_stalking.get("person_rate", 0) >= 0.35
    stalking_not_fight_like = peak_stalking.get("close_motion_rate", 0) < 0.45 and peak_stalking.get("swap_rate", 0) < 0.25

    if fight_score >= FIGHT_THRESHOLD and fight_multi_ok and fight_interaction_ok:
        label = "fight"
        reason = (
            "영상 전체 평균이 아니라 가장 강한 짧은 구간에서, "
            "근접 상태의 움직임/접근-분리/방향 변화/위치 뒤섞임이 함께 나타나 싸움 후보로 판단"
        )
    elif stalking_score >= STALKING_THRESHOLD and stalking_person_ok and stalking_not_fight_like:
        label = "stalking"
        reason = (
            "사람이 지속적으로 등장하고, 싸움처럼 격한 근접 움직임보다는 "
            "이동 흐름 또는 따라가는 형태가 더 강해 스토킹 후보로 판단"
        )
    else:
        label = "other"
        reason = (
            "짧은 구간 기준 peak score를 보아도 fight/stalking 조건을 충분히 만족하지 않아 기타로 판단"
        )

    return {
        "label": label,
        "reason": reason,
        "scores": {
            "peak_fight_score": round(fight_score, 4),
            "peak_stalking_score": round(stalking_score, 4),
            "overall_fight_score": overall.get("fight_score", 0),
            "overall_stalking_score": overall.get("stalking_score", 0),
        },
        "peak_fight_window": peak_fight,
        "peak_stalking_window": peak_stalking,
        "overall_features": overall,
    }


In [8]:

# Cell 8. improved bd4 행동 판정 실행 및 저장
aa_json_paths = sorted(AA_FEATURE_DIR.rglob("*.json"))
if MAX_JSON_FILES is not None:
    aa_json_paths = aa_json_paths[:MAX_JSON_FILES]

bd4_results = []
bd4_failed = []

for json_path in tqdm(aa_json_paths):
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            aa_data = json.load(f)
        pred = classify_action_from_aa(aa_data)
        item = {
            "video_name": aa_data.get("video_name"),
            "video_path": aa_data.get("video_path"),
            "relative_video_path": aa_data.get("relative_video_path"),
            **pred,
        }
        bd4_results.append(item)
        out_path = AA_BD4_DIR / json_path.name
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(item, f, ensure_ascii=False, indent=2)
    except Exception as e:
        bd4_failed.append({"json": str(json_path), "error": repr(e)})

bd4_summary = {
    "task": "ot4_bd4_improved_action_classification",
    "input_dir": str(AA_FEATURE_DIR),
    "output_dir": str(AA_BD4_DIR),
    "total": len(aa_json_paths),
    "success": len(bd4_results),
    "failed": len(bd4_failed),
    "label_counts": dict(Counter(r["label"] for r in bd4_results)),
    "results": bd4_results,
    "failed_items": bd4_failed,
}

bd4_summary_path = AA_BD4_DIR / "bd4_improved_summary.json"
with open(bd4_summary_path, "w", encoding="utf-8") as f:
    json.dump(bd4_summary, f, ensure_ascii=False, indent=2)

print("\n================================")
print("bd4 improved 판정 완료")
print("전체:", bd4_summary["total"])
print("성공:", bd4_summary["success"])
print("실패:", bd4_summary["failed"])
print("label_counts:", bd4_summary["label_counts"])
print("summary 저장:", bd4_summary_path)
print("================================")


100%|█████████████████████████████████████████████████████████████████████████████████| 22/22 [00:00<00:00, 203.77it/s]


bd4 improved 판정 완료
전체: 22
성공: 22
실패: 0
label_counts: {'stalking': 9, 'fight': 4, 'other': 9}
summary 저장: C:\Users\우성한\outputs\prediction\aa\bd4_improved\bd4_improved_summary.json


In [9]:

# Cell 9. cf6-like 이륜차 판정 함수
# 주의: 이 셀은 기존 cf6의 무거운 segmentation/helmet 모델 대신,
# CF2와 aa_features에 있는 객체/움직임 정보를 기반으로 빠르게 동시 태그를 붙입니다.
# 기존 cf6처럼 SegFormer/헬멧 모델을 붙이고 싶으면, 아래 결과 구조에 sidewalk_status/helmet_status만 교체하면 됩니다.


def person_vehicle_overlap(person_bbox, vehicle_bbox):
    return bbox_iou(person_bbox, vehicle_bbox)


def classify_two_wheeler_from_aa(aa_data):
    frames = aa_data.get("frames", [])
    vehicle_frames = 0
    rider_like_frames = 0
    person_vehicle_close_frames = 0
    vehicle_classes = Counter()

    for fr in frames:
        vehicles = fr.get("vehicles", [])
        persons = fr.get("persons", [])
        if vehicles:
            vehicle_frames += 1
            for v in vehicles:
                vehicle_classes[v.get("class", "unknown")] += 1

        # 이륜차와 사람이 가까이 있거나 bbox가 조금이라도 겹치면 rider-like로 봄
        rider_like = False
        close_like = False
        for v in vehicles:
            vc = v.get("center")
            for p in persons:
                pc = p.get("center")
                if vc and pc and dist(vc, pc) < 260:
                    close_like = True
                if p.get("bbox") and v.get("bbox") and person_vehicle_overlap(p["bbox"], v["bbox"]) > 0.005:
                    rider_like = True

        if close_like:
            person_vehicle_close_frames += 1
        if rider_like or close_like:
            rider_like_frames += 1

    total = len(frames) if frames else 1
    vehicle_ratio = vehicle_frames / total
    rider_like_ratio = rider_like_frames / total
    close_ratio = person_vehicle_close_frames / total

    two_wheeler_detected = vehicle_ratio >= 0.03 or vehicle_frames >= 2
    rider_detected = rider_like_ratio >= 0.03 or rider_like_frames >= 2

    # 지금 단계에서는 segmentation/helmet 없이 알 수 없는 값으로 둔다.
    sidewalk_status = "unknown"
    helmet_status = "unknown"

    tags = []
    if two_wheeler_detected:
        tags.append("two_wheeler_detected")
    if rider_detected:
        tags.append("rider_like")

    # label은 현재 확실히 판정 가능한 범위만 부여
    if two_wheeler_detected and rider_detected:
        label = "two_wheeler_rider_like"
    elif two_wheeler_detected:
        label = "two_wheeler_detected"
    else:
        label = "no_two_wheeler"

    return {
        "label": label,
        "tags": tags,
        "two_wheeler_detected": two_wheeler_detected,
        "rider_detected": rider_detected,
        "sidewalk_status": sidewalk_status,
        "helmet_status": helmet_status,
        "features": {
            "total_sampled_frames": total,
            "vehicle_frames": vehicle_frames,
            "rider_like_frames": rider_like_frames,
            "person_vehicle_close_frames": person_vehicle_close_frames,
            "vehicle_ratio": round(vehicle_ratio, 4),
            "rider_like_ratio": round(rider_like_ratio, 4),
            "person_vehicle_close_ratio": round(close_ratio, 4),
            "vehicle_classes": dict(vehicle_classes),
        },
        "reason": "CF2 객체 탐지와 사람-이륜차 근접/겹침을 기준으로 이륜차 관련 여부를 동시 태그로 판단",
    }


In [10]:

# Cell 10. cf6-like 이륜차 판정 실행 및 저장
cf6_results = []
cf6_failed = []

for json_path in tqdm(aa_json_paths):
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            aa_data = json.load(f)
        pred = classify_two_wheeler_from_aa(aa_data)
        item = {
            "video_name": aa_data.get("video_name"),
            "video_path": aa_data.get("video_path"),
            "relative_video_path": aa_data.get("relative_video_path"),
            **pred,
        }
        cf6_results.append(item)
        out_path = AA_CF6_DIR / json_path.name
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(item, f, ensure_ascii=False, indent=2)
    except Exception as e:
        cf6_failed.append({"json": str(json_path), "error": repr(e)})

cf6_summary = {
    "task": "ot4_cf6_like_two_wheeler_classification",
    "input_dir": str(AA_FEATURE_DIR),
    "output_dir": str(AA_CF6_DIR),
    "total": len(aa_json_paths),
    "success": len(cf6_results),
    "failed": len(cf6_failed),
    "label_counts": dict(Counter(r["label"] for r in cf6_results)),
    "tag_counts": dict(Counter(t for r in cf6_results for t in r.get("tags", []))),
    "results": cf6_results,
    "failed_items": cf6_failed,
}

cf6_summary_path = AA_CF6_DIR / "cf6_like_summary.json"
with open(cf6_summary_path, "w", encoding="utf-8") as f:
    json.dump(cf6_summary, f, ensure_ascii=False, indent=2)

print("\n================================")
print("cf6-like 이륜차 판정 완료")
print("전체:", cf6_summary["total"])
print("성공:", cf6_summary["success"])
print("실패:", cf6_summary["failed"])
print("label_counts:", cf6_summary["label_counts"])
print("tag_counts:", cf6_summary["tag_counts"])
print("summary 저장:", cf6_summary_path)
print("================================")


100%|█████████████████████████████████████████████████████████████████████████████████| 22/22 [00:00<00:00, 366.78it/s]


cf6-like 이륜차 판정 완료
전체: 22
성공: 22
실패: 0
label_counts: {'two_wheeler_rider_like': 8, 'no_two_wheeler': 14}
tag_counts: {'two_wheeler_detected': 8, 'rider_like': 8}
summary 저장: C:\Users\우성한\outputs\prediction\aa\cf6_like\cf6_like_summary.json


In [11]:

# Cell 11. ot4 최종 동시 통합 결과 생성
bd4_by_rel = {r.get("relative_video_path"): r for r in bd4_results}
cf6_by_rel = {r.get("relative_video_path"): r for r in cf6_results}

all_keys = sorted(set(bd4_by_rel) | set(cf6_by_rel))
final_results = []

for rel in all_keys:
    bd4 = bd4_by_rel.get(rel)
    cf6 = cf6_by_rel.get(rel)

    combined_tags = []
    if bd4:
        combined_tags.append(bd4["label"])
    if cf6:
        combined_tags.extend(cf6.get("tags", []))

    # 우선순위 없이 나란히 표기
    bd4_label = bd4["label"] if bd4 else "missing_bd4"
    cf6_label = cf6["label"] if cf6 else "missing_cf6"
    parallel_label = f"{bd4_label} + {cf6_label}"

    item = {
        "video_name": (bd4 or cf6 or {}).get("video_name"),
        "video_path": (bd4 or cf6 or {}).get("video_path"),
        "relative_video_path": rel,
        "parallel_label": parallel_label,
        "combined_tags": sorted(set(combined_tags)),
        "bd4_label": bd4_label,
        "cf6_label": cf6_label,
        "bd4_action": bd4,
        "cf6_two_wheeler": cf6,
    }
    final_results.append(item)

final_summary = {
    "task": "ot4_parallel_action_and_two_wheeler_with_aa_features",
    "video_root": str(VIDEO_ROOT),
    "aa_dir": str(AA_DIR),
    "cf2_dir": str(AA_CF2_DIR),
    "aa_feature_dir": str(AA_FEATURE_DIR),
    "bd4_dir": str(AA_BD4_DIR),
    "cf6_dir": str(AA_CF6_DIR),
    "final_dir": str(AA_FINAL_DIR),
    "total": len(final_results),
    "bd4_label_counts": dict(Counter(r["bd4_label"] for r in final_results)),
    "cf6_label_counts": dict(Counter(r["cf6_label"] for r in final_results)),
    "parallel_label_counts": dict(Counter(r["parallel_label"] for r in final_results)),
    "combined_tag_counts": dict(Counter(t for r in final_results for t in r.get("combined_tags", []))),
    "results": final_results,
}

final_summary_path = AA_FINAL_DIR / "ot4_final_summary.json"
with open(final_summary_path, "w", encoding="utf-8") as f:
    json.dump(final_summary, f, ensure_ascii=False, indent=2)

print("\n================================")
print("ot4 최종 동시 통합 완료")
print("전체:", final_summary["total"])
print("bd4_label_counts:", final_summary["bd4_label_counts"])
print("cf6_label_counts:", final_summary["cf6_label_counts"])
print("combined_tag_counts:", final_summary["combined_tag_counts"])
print("summary 저장:", final_summary_path)
print("================================")



ot4 최종 동시 통합 완료
전체: 22
bd4_label_counts: {'stalking': 9, 'fight': 4, 'other': 9}
cf6_label_counts: {'two_wheeler_rider_like': 8, 'no_two_wheeler': 14}
combined_tag_counts: {'rider_like': 8, 'stalking': 9, 'two_wheeler_detected': 8, 'fight': 4, 'other': 9}
summary 저장: C:\Users\우성한\outputs\prediction\aa\final_ot4\ot4_final_summary.json


In [12]:

# Cell 12. 결과 출력 및 검색 함수

def print_ot4_by_label(label=None, limit=50):
    shown = 0
    for r in final_results:
        tags = set(r.get("combined_tags", []))
        if label is not None and label not in tags and r.get("bd4_label") != label and r.get("cf6_label") != label:
            continue
        print("=" * 80)
        print("video_name:", r.get("video_name"))
        print("relative_video_path:", r.get("relative_video_path"))
        print("parallel_label:", r.get("parallel_label"))
        print("combined_tags:", r.get("combined_tags"))
        bd4 = r.get("bd4_action") or {}
        print("bd4_scores:", bd4.get("scores"))
        pfw = bd4.get("peak_fight_window") or {}
        print("peak_fight_window:", {
            "start_sec": pfw.get("start_sec"),
            "end_sec": pfw.get("end_sec"),
            "fight_score": pfw.get("fight_score"),
            "close_motion_rate": pfw.get("close_motion_rate"),
            "approach_separate_rate": pfw.get("approach_separate_rate"),
            "swap_rate": pfw.get("swap_rate"),
            "same_direction_rate": pfw.get("same_direction_rate"),
        })
        cf6 = r.get("cf6_two_wheeler") or {}
        print("cf6_features:", cf6.get("features"))
        print()
        shown += 1
        if shown >= limit:
            break
    print("shown:", shown)


def search_ot4_results(query, limit=50):
    q = str(query).lower()
    aliases = {
        "싸움": ["fight"],
        "스토킹": ["stalking"],
        "기타": ["other"],
        "이륜차": ["two_wheeler"],
        "자전거": ["bicycle", "two_wheeler"],
        "오토바이": ["motorcycle", "two_wheeler"],
        "라이더": ["rider_like"],
    }
    keys = [q]
    for k, vals in aliases.items():
        if k in q:
            keys.extend(vals)

    matched = []
    for r in final_results:
        hay = json.dumps(r, ensure_ascii=False).lower()
        if all(any(key in hay for key in keys_part.split()) for keys_part in [" ".join(keys)]):
            matched.append(r)
        else:
            if any(key in hay for key in keys):
                matched.append(r)

    print(f"query={query!r}, matched={len(matched)}")
    for r in matched[:limit]:
        print("=" * 80)
        print("video_name:", r.get("video_name"))
        print("relative_video_path:", r.get("relative_video_path"))
        print("parallel_label:", r.get("parallel_label"))
        print("combined_tags:", r.get("combined_tags"))
        bd4 = r.get("bd4_action") or {}
        print("bd4_scores:", bd4.get("scores"))
    return matched


print("예시:")
print('print_ot4_by_label("fight")')
print('search_ot4_results("싸움")')
print('search_ot4_results("이륜차")')
print('search_ot4_results("싸움 이륜차")')


예시:
print_ot4_by_label("fight")
search_ot4_results("싸움")
search_ot4_results("이륜차")
search_ot4_results("싸움 이륜차")


In [13]:

# Cell 13. 싸움 점수 튜닝용 요약표
# fight가 너무 많이 잡히면 FIGHT_THRESHOLD를 올리거나 close_motion/approach/swap 조건을 강화하면 됨.
# 진짜 싸움이 낮게 잡히면 FIGHT_THRESHOLD를 조금 낮추되, 단순 close_rate가 아니라 peak_fight_score 기준으로 확인.

fight_rows = []
for r in final_results:
    bd4 = r.get("bd4_action") or {}
    pfw = bd4.get("peak_fight_window") or {}
    fight_rows.append({
        "video_name": r.get("video_name"),
        "relative_video_path": r.get("relative_video_path"),
        "bd4_label": r.get("bd4_label"),
        "cf6_label": r.get("cf6_label"),
        "peak_fight_score": pfw.get("fight_score"),
        "start_sec": pfw.get("start_sec"),
        "end_sec": pfw.get("end_sec"),
        "multi_rate": pfw.get("multi_rate"),
        "close_rate": pfw.get("close_rate"),
        "close_motion_rate": pfw.get("close_motion_rate"),
        "approach_separate_rate": pfw.get("approach_separate_rate"),
        "overlap_rate": pfw.get("overlap_rate"),
        "swap_rate": pfw.get("swap_rate"),
        "same_direction_rate": pfw.get("same_direction_rate"),
    })

fight_rows_sorted = sorted(fight_rows, key=lambda x: x.get("peak_fight_score") or 0, reverse=True)

for row in fight_rows_sorted[:30]:
    print(row)

score_table_path = AA_FINAL_DIR / "ot4_fight_score_table.json"
with open(score_table_path, "w", encoding="utf-8") as f:
    json.dump(fight_rows_sorted, f, ensure_ascii=False, indent=2)

print("saved:", score_table_path)


{'video_name': 'md_e2122_c1.mp4', 'relative_video_path': '이륜 이동수단 헬멧 미착용\\이륜 이동수단 운전자 1인 헬멧 미착용\\md_e2122\\md_e2122_c1.mp4', 'bd4_label': 'fight', 'cf6_label': 'two_wheeler_rider_like', 'peak_fight_score': 0.5846, 'start_sec': 13.333, 'end_sec': 18.0, 'multi_rate': 1.0, 'close_rate': 1.0, 'close_motion_rate': 0.4, 'approach_separate_rate': 0.2667, 'overlap_rate': 0.8, 'swap_rate': 0.6, 'same_direction_rate': 0.8}
{'video_name': 'md_e2122_c2.mp4', 'relative_video_path': '이륜 이동수단 헬멧 미착용\\이륜 이동수단 운전자 1인 헬멧 미착용\\md_e2122\\md_e2122_c2.mp4', 'bd4_label': 'fight', 'cf6_label': 'no_two_wheeler', 'peak_fight_score': 0.4865, 'start_sec': 24.0, 'end_sec': 28.667, 'multi_rate': 0.9333, 'close_rate': 0.9333, 'close_motion_rate': 0.2, 'approach_separate_rate': 0.1333, 'overlap_rate': 0.6, 'swap_rate': 0.6, 'same_direction_rate': 0.4}
{'video_name': 'ph_e2063_c1.mp4', 'relative_video_path': '싸움\\신체적 충돌을 동반한 싸움\\ph_e2063\\ph_e2063_c1.mp4', 'bd4_label': 'fight', 'cf6_label': 'no_two_wheeler', 'peak_fig

In [14]:
# ============================================================
# OT4 결과 한눈에 보기 셀
# ============================================================

import json
from pathlib import Path
from collections import Counter

AA_ROOT = Path(r"C:\Users\우성한\outputs\prediction\aa")
FINAL_DIR = AA_ROOT / "final_ot4"

summary_path = FINAL_DIR / "ot4_final_summary.json"
score_table_path = FINAL_DIR / "ot4_fight_score_table.json"

if not summary_path.exists():
    raise FileNotFoundError(f"summary 파일이 없습니다: {summary_path}")

with open(summary_path, "r", encoding="utf-8") as f:
    summary = json.load(f)

results = summary.get("results", [])

print("=" * 80)
print("OT4 최종 결과 요약")
print("=" * 80)
print("summary 파일:", summary_path)
print("전체 영상 수:", len(results))

# ------------------------------------------------------------
# 1. 최종 라벨 개수
# ------------------------------------------------------------

final_labels = [r.get("final_label", "unknown") for r in results]
bd4_labels = [r.get("bd4_label", "unknown") for r in results]
cf6_labels = [r.get("cf6_label", "unknown") for r in results]

print()
print("=" * 80)
print("1. 최종 라벨별 개수")
print("=" * 80)

for label, count in Counter(final_labels).most_common():
    print(f"{label:35s} : {count}")

print()
print("=" * 80)
print("2. 행동 판정 bd4_improved 기준 개수")
print("=" * 80)

for label, count in Counter(bd4_labels).most_common():
    print(f"{label:35s} : {count}")

print()
print("=" * 80)
print("3. 이륜차 cf6_like 기준 개수")
print("=" * 80)

for label, count in Counter(cf6_labels).most_common():
    print(f"{label:35s} : {count}")

# ------------------------------------------------------------
# 2. 영상별 핵심 결과 표
# ------------------------------------------------------------

print()
print("=" * 80)
print("4. 영상별 최종 판정")
print("=" * 80)

for idx, r in enumerate(results, start=1):
    video_name = r.get("video_name", "")
    rel_path = r.get("relative_video_path", "")
    final_label = r.get("final_label", "unknown")
    bd4_label = r.get("bd4_label", "unknown")
    cf6_label = r.get("cf6_label", "unknown")
    tags = r.get("combined_tags", [])

    aa = r.get("aa_features", {})
    peak_fight_score = aa.get("peak_fight_score", None)
    peak_stalking_score = aa.get("peak_stalking_score", None)
    best_window = aa.get("best_fight_window", {})

    print()
    print("-" * 80)
    print(f"[{idx}] {video_name}")
    if rel_path:
        print("상대 경로:", rel_path)

    print("최종 판정:", final_label)
    print("행동 판정:", bd4_label)
    print("이륜차 판정:", cf6_label)
    print("태그:", ", ".join(tags) if tags else "없음")

    if peak_fight_score is not None:
        print("최고 싸움 점수:", round(peak_fight_score, 4))

    if peak_stalking_score is not None:
        print("최고 스토킹 점수:", round(peak_stalking_score, 4))

    if best_window:
        start_t = best_window.get("start_time")
        end_t = best_window.get("end_time")
        if start_t is not None and end_t is not None:
            print(f"싸움 의심 구간: {start_t:.1f}초 ~ {end_t:.1f}초")

    reason = r.get("final_reason") or r.get("reason")
    if reason:
        print("판정 이유:", reason)

# ------------------------------------------------------------
# 3. 싸움 점수 높은 순서
# ------------------------------------------------------------

print()
print("=" * 80)
print("5. 싸움 점수 높은 순 TOP 20")
print("=" * 80)

fight_rank = []

for r in results:
    aa = r.get("aa_features", {})
    fight_score = aa.get("peak_fight_score", 0)

    fight_rank.append({
        "video_name": r.get("video_name"),
        "relative_video_path": r.get("relative_video_path"),
        "final_label": r.get("final_label"),
        "bd4_label": r.get("bd4_label"),
        "cf6_label": r.get("cf6_label"),
        "peak_fight_score": fight_score,
        "best_fight_window": aa.get("best_fight_window", {})
    })

fight_rank = sorted(
    fight_rank,
    key=lambda x: x.get("peak_fight_score", 0),
    reverse=True
)

for idx, item in enumerate(fight_rank[:20], start=1):
    w = item.get("best_fight_window", {})
    start_t = w.get("start_time")
    end_t = w.get("end_time")

    print()
    print(f"{idx}. {item.get('video_name')}")
    print("   상대 경로:", item.get("relative_video_path"))
    print("   최종 판정:", item.get("final_label"))
    print("   행동 판정:", item.get("bd4_label"))
    print("   이륜차 판정:", item.get("cf6_label"))
    print("   최고 싸움 점수:", round(item.get("peak_fight_score", 0), 4))

    if start_t is not None and end_t is not None:
        print(f"   의심 구간: {start_t:.1f}초 ~ {end_t:.1f}초")

# ------------------------------------------------------------
# 4. 겹침 상황만 따로 출력
# ------------------------------------------------------------

print()
print("=" * 80)
print("6. 행동 이상 + 이륜차 관련이 동시에 잡힌 영상")
print("=" * 80)

overlap = []

for r in results:
    bd4_label = r.get("bd4_label")
    cf6_label = r.get("cf6_label")
    tags = r.get("combined_tags", [])

    has_action = bd4_label in ["fight", "stalking"]
    has_two_wheeler = any("two_wheeler" in str(t) for t in tags) or (
        cf6_label is not None and "two_wheeler" in str(cf6_label)
    )

    if has_action and has_two_wheeler:
        overlap.append(r)

if not overlap:
    print("겹치는 영상 없음")
else:
    for idx, r in enumerate(overlap, start=1):
        print()
        print(f"{idx}. {r.get('video_name')}")
        print("   상대 경로:", r.get("relative_video_path"))
        print("   행동 판정:", r.get("bd4_label"))
        print("   이륜차 판정:", r.get("cf6_label"))
        print("   최종 판정:", r.get("final_label"))
        print("   태그:", ", ".join(r.get("combined_tags", [])))

# ------------------------------------------------------------
# 5. 라벨별 파일명만 간단 출력
# ------------------------------------------------------------

print()
print("=" * 80)
print("7. 라벨별 파일명 목록")
print("=" * 80)

labels_to_show = sorted(set(final_labels))

for label in labels_to_show:
    print()
    print(f"[{label}]")
    matched = [r for r in results if r.get("final_label", "unknown") == label]

    if not matched:
        print("  없음")
        continue

    for r in matched:
        print(" -", r.get("video_name"), "|", r.get("relative_video_path"))

# ------------------------------------------------------------
# 6. 결과를 txt로도 저장
# ------------------------------------------------------------

txt_path = FINAL_DIR / "ot4_readable_result.txt"

lines = []
lines.append("=" * 80)
lines.append("OT4 최종 결과 요약")
lines.append("=" * 80)
lines.append(f"전체 영상 수: {len(results)}")
lines.append("")

lines.append("[최종 라벨별 개수]")
for label, count in Counter(final_labels).most_common():
    lines.append(f"{label:35s} : {count}")

lines.append("")
lines.append("[영상별 최종 판정]")

for idx, r in enumerate(results, start=1):
    aa = r.get("aa_features", {})
    w = aa.get("best_fight_window", {})

    lines.append("-" * 80)
    lines.append(f"[{idx}] {r.get('video_name')}")
    lines.append(f"상대 경로: {r.get('relative_video_path')}")
    lines.append(f"최종 판정: {r.get('final_label')}")
    lines.append(f"행동 판정: {r.get('bd4_label')}")
    lines.append(f"이륜차 판정: {r.get('cf6_label')}")
    lines.append(f"태그: {', '.join(r.get('combined_tags', []))}")

    if "peak_fight_score" in aa:
        lines.append(f"최고 싸움 점수: {aa.get('peak_fight_score')}")
    if "peak_stalking_score" in aa:
        lines.append(f"최고 스토킹 점수: {aa.get('peak_stalking_score')}")

    start_t = w.get("start_time")
    end_t = w.get("end_time")
    if start_t is not None and end_t is not None:
        lines.append(f"싸움 의심 구간: {start_t:.1f}초 ~ {end_t:.1f}초")

    reason = r.get("final_reason") or r.get("reason")
    if reason:
        lines.append(f"판정 이유: {reason}")

with open(txt_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print()
print("=" * 80)
print("TXT 요약 저장 완료")
print("=" * 80)
print(txt_path)

OT4 최종 결과 요약
summary 파일: C:\Users\우성한\outputs\prediction\aa\final_ot4\ot4_final_summary.json
전체 영상 수: 22

1. 최종 라벨별 개수
unknown                             : 22

2. 행동 판정 bd4_improved 기준 개수
stalking                            : 9
other                               : 9
fight                               : 4

3. 이륜차 cf6_like 기준 개수
no_two_wheeler                      : 14
two_wheeler_rider_like              : 8

4. 영상별 최종 판정

--------------------------------------------------------------------------------
[1] lo_e1281_c1.mp4
상대 경로: 스토킹\특정 구역 내 지속 배회\lo_e1281\lo_e1281_c1.mp4
최종 판정: unknown
행동 판정: stalking
이륜차 판정: two_wheeler_rider_like
태그: rider_like, stalking, two_wheeler_detected

--------------------------------------------------------------------------------
[2] lo_e1281_c2.mp4
상대 경로: 스토킹\특정 구역 내 지속 배회\lo_e1281\lo_e1281_c2.mp4
최종 판정: unknown
행동 판정: stalking
이륜차 판정: two_wheeler_rider_like
태그: rider_like, stalking, two_wheeler_detected

--------------------------------------------------